# Submission Notebook

This notebook generates the final competition submission:
- Runs the complete pipeline on test data
- Generates CTC-formatted output
- Creates the submission CSV file

In [1]:
import os
import sys
import numpy as np
from pathlib import Path
import pandas as pd
import logging

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / "src"))

logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)

VOXEL_SIZE_UM = (1.625, 0.40625, 0.40625)

print("=" * 60)
print("SETUP INSTRUCTIONS")
print("=" * 60)
print("\nTo run this notebook, execute from project root:")
print("\n  pip install -e .")
print("  pip install zarr pandas")
print("\nThen restart the Jupyter kernel.")
print("=" * 60)

# Try importing modules
try:
    from biohub_tracking.data.zarr_loader import iter_frames
    from biohub_tracking.segmentation.segmenter import CellSegmenter
    from biohub_tracking.tracking.linker import HungarianLinker
    from biohub_tracking.tracking.division_classifier import DivisionClassifier
    from biohub_tracking.evaluation.submission_builder import SubmissionBuilder
    MODULES_AVAILABLE = True
    print("✓ biohub_tracking modules available")
except ImportError as e:
    MODULES_AVAILABLE = False
    print(f"✗ biohub_tracking not available - {e}")

SETUP INSTRUCTIONS

To run this notebook, execute from project root:

  pip install -e .
  pip install zarr pandas

Then restart the Jupyter kernel.
✗ biohub_tracking not available - No module named 'biohub_tracking'


## 1. Configuration

In [ ]:
# Configuration for submission
config = {
    'test_dir': Path('../data/test'),
    'output_dir': Path('../outputs'),
    'submission_path': Path('submission.csv'),
    
    'segmentation': {
        'method': 'blob',
        'min_size': 30,
        'anisotropy': VOXEL_SIZE_UM[0] / VOXEL_SIZE_UM[1],
        'voxel_size_um': VOXEL_SIZE_UM,
    },
    
    'tracking': {
        'max_distance': 7.0,
        'use_volume_cost': False,
    },
    
    'division': {
        'max_distance_um': 10.0,
    }
}

config['output_dir'].mkdir(parents=True, exist_ok=True)

print("Submission Configuration:")
print("="*50)
print(f"Test directory: {config['test_dir']}")
print(f"Output directory: {config['output_dir']}")
print(f"Submission file: {config['submission_path']}")
print(f"\nSegmentation method: {config['segmentation']['method']}")
print(f"Min cell size: {config['segmentation']['min_size']} voxels")
print(f"Tracking max distance: {config['tracking']['max_distance']} µm")

## 2. Define Processing Function

In [ ]:
def process_sample(sample_path: Path, segmenter: CellSegmenter, linker: HungarianLinker, 
                   division_detector: DivisionClassifier):
    """Process a single sample through the full pipeline."""
    
    all_cells = {}
    for frame_index, image in iter_frames(sample_path):
        labels, cells = segmenter.segment_frame(image, frame_index=frame_index)
        all_cells[frame_index] = cells
    
    links = []
    linked_ids = {}
    
    sorted_frames = sorted(all_cells.keys())
    for i in range(len(sorted_frames) - 1):
        frame1 = sorted_frames[i]
        frame2 = sorted_frames[i + 1]
        
        frame_links = linker.link(all_cells[frame1], all_cells[frame2])
        links.extend([
            (frame1, source, target, confidence)
            for source, target, confidence in frame_links
        ])
        linked_ids[frame1] = [(source, target) for source, target, _ in frame_links]
    
    divisions = division_detector.detect(all_cells, linked_ids)
    return all_cells, links, divisions

print("Processing function defined.")

## 3. Generate Submission

In [ ]:
segmenter = CellSegmenter(**config['segmentation'])
linker = HungarianLinker(**config['tracking'])
division_detector = DivisionClassifier(**config['division'])
builder = SubmissionBuilder()

print("Components initialized.")

if not config['test_dir'].exists():
    print(f"\nTest directory not found at {config['test_dir']}")
    print("Demo mode: Creating sample submission")
    rows = []
else:
    samples = sorted(config['test_dir'].glob('*.zarr'))
    print(f"\nFound {len(samples)} samples to process")
    
    rows = []
    for sample_idx, sample_path in enumerate(samples):
        logger.info(f"Processing {sample_path.name}")
        all_cells, links, divisions = process_sample(
            sample_path, segmenter, linker, division_detector
        )
        sample_rows = builder.build_rows(
            sample_path.stem, all_cells, links, divisions
        )
        rows.extend(sample_rows)
        logger.info(f"Generated {len(sample_rows)} rows")

print(f"Total rows: {len(rows)}")

## 4. Preview Submission

In [ ]:
if rows:
    df = pd.DataFrame(rows)
    
    print("Submission Preview:")
    print(df.head(10))
    print(f"\nShape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nStatistics:")
    print(f"  Unique samples: {df['TRA'].nunique() if 'TRA' in df.columns else 'N/A'}")
    print(f"  Total rows: {len(df)}")
else:
    print("No data to display")

## 5. Write Submission File

In [ ]:
try:
    builder.write(rows, config['submission_path'])
    
    if config['submission_path'].exists():
        file_size = config['submission_path'].stat().st_size
        line_count = sum(1 for line in open(config['submission_path']))
        
        print(f"Submission Created Successfully!")
        print(f"Path: {config['submission_path'].absolute()}")
        print(f"Size: {file_size / 1024:.1f} KB")
        print(f"Lines: {line_count}")
    else:
        print("Error: File not created")
        
except Exception as e:
    print(f"Error: {e}")

## 6. Summary

In [ ]:
print("\nFinal Summary:")
print("="*50)
print(f"Segmentation: {config['segmentation']['method']}")
print(f"Tracking: Hungarian matching")
print(f"Max distance: {config['tracking']['max_distance']} µm")
print(f"Voxel scale: {VOXEL_SIZE_UM} µm (Z, Y, X)")
print(f"\nSubmission: {config['submission_path']}")
print("Ready for Kaggle submission!")